In [1]:
# DM4ML -Assignment - Ingestion - API

import json
import time
import random
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests

# ============================================================
# RecoMart - DummyJSON API Ingestion Script
# - Fetches product + category data from DummyJSON
# - Retries on transient failures
# - Falls back to embedded sample data if API is unreachable
# - Stores raw JSON, bronze parquet, logs, and manifest
# ============================================================

# -----------------------------
# 1) CONFIG
# -----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "data"
SOURCE_NAME = "dummyjson"

USE_OFFLINE_FALLBACK = True
MAX_ATTEMPTS = 5
REQUEST_TIMEOUT = 30
PAGE_SIZE = 30

UTC_NOW = datetime.now(timezone.utc)
LOAD_DATE = UTC_NOW.strftime("%Y-%m-%d")
LOAD_HOUR = UTC_NOW.strftime("%H")
BATCH_ID = UTC_NOW.strftime("%Y%m%dT%H%M%SZ")
INGESTION_TS = UTC_NOW.isoformat()

BASE_URL = "https://dummyjson.com"
PRODUCTS_ENDPOINT = f"{BASE_URL}/products"
CATEGORIES_ENDPOINT = f"{BASE_URL}/products/categories"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / SOURCE_NAME / f"load_date={LOAD_DATE}" / f"load_hour={LOAD_HOUR}"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / SOURCE_NAME
LOG_DIR = PROJECT_ROOT / "logs"
METADATA_DIR = PROJECT_ROOT / "metadata"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / f"{SOURCE_NAME}_ingestion_log.jsonl"
MANIFEST_FILE = METADATA_DIR / f"{SOURCE_NAME}_manifest_{BATCH_ID}.json"

# -----------------------------
# 2) LOGGING
# -----------------------------
def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "source": SOURCE_NAME,
        "stage": stage,
        "status": status,
        "message": message,
        "batch_id": BATCH_ID,
    }
    if extra:
        record.update(extra)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

# -----------------------------
# 3) OFFLINE FALLBACK DATA
# -----------------------------
def get_fallback_products():
    return {
        "products": [
            {
                "id": 1,
                "title": "Essence Mascara Lash Princess",
                "description": "Lengthening mascara for daily wear",
                "category": "beauty",
                "price": 9.99,
                "discountPercentage": 7.17,
                "rating": 4.94,
                "stock": 5,
                "brand": "Essence",
                "sku": "BEAUTY-001",
                "weight": 20,
                "dimensions": {"width": 2.0, "height": 12.0, "depth": 2.0},
                "warrantyInformation": "6 months",
                "shippingInformation": "Ships in 2 days",
                "availabilityStatus": "Low Stock",
                "reviews": [],
                "returnPolicy": "7 days return",
                "minimumOrderQuantity": 1,
            },
            {
                "id": 2,
                "title": "iPhone 15 Case",
                "description": "Protective slim case",
                "category": "smartphones",
                "price": 19.99,
                "discountPercentage": 5.0,
                "rating": 4.5,
                "stock": 50,
                "brand": "AppleGear",
                "sku": "PHONE-002",
                "weight": 50,
                "dimensions": {"width": 8.0, "height": 15.0, "depth": 1.0},
                "warrantyInformation": "12 months",
                "shippingInformation": "Ships in 1 day",
                "availabilityStatus": "In Stock",
                "reviews": [],
                "returnPolicy": "14 days return",
                "minimumOrderQuantity": 1,
            },
            {
                "id": 3,
                "title": "Running Shoes",
                "description": "Comfortable sports shoes",
                "category": "mens-shoes",
                "price": 79.99,
                "discountPercentage": 10.0,
                "rating": 4.3,
                "stock": 22,
                "brand": "SprintX",
                "sku": "SHOE-003",
                "weight": 700,
                "dimensions": {"width": 12.0, "height": 10.0, "depth": 32.0},
                "warrantyInformation": "3 months",
                "shippingInformation": "Ships in 3 days",
                "availabilityStatus": "In Stock",
                "reviews": [],
                "returnPolicy": "10 days return",
                "minimumOrderQuantity": 1,
            },
        ],
        "total": 3,
        "skip": 0,
        "limit": 30,
    }

def get_fallback_categories():
    return ["beauty", "smartphones", "mens-shoes"]

# -----------------------------
# 4) HTTP FETCH
# -----------------------------
def fetch_json(url, params=None, timeout=30, max_attempts=5):
    last_error = None

    headers = {
        "User-Agent": "RecoMart-Assignment/1.0",
        "Accept": "application/json",
        "Connection": "close",
    }

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            last_error = str(e)

            log_event(
                "api_request",
                "failed",
                f"Request failed on attempt {attempt}",
                extra={"url": url, "params": params, "error": last_error},
            )

            if attempt < max_attempts:
                sleep_seconds = (2 ** attempt) + random.uniform(0.5, 1.5)
                time.sleep(sleep_seconds)

    raise RuntimeError(f"API request failed after {max_attempts} attempts: {last_error}")

def fetch_all_products(page_size=30):
    log_event("api_fetch_products", "started", "Fetching products from DummyJSON")

    first_page = fetch_json(PRODUCTS_ENDPOINT, params={"limit": page_size, "skip": 0}, timeout=REQUEST_TIMEOUT, max_attempts=MAX_ATTEMPTS)
    products = first_page.get("products", [])
    total = first_page.get("total", len(products))

    all_products = list(products)
    skip = len(products)

    while skip < total:
        page = fetch_json(PRODUCTS_ENDPOINT, params={"limit": page_size, "skip": skip}, timeout=REQUEST_TIMEOUT, max_attempts=MAX_ATTEMPTS)
        page_products = page.get("products", [])
        all_products.extend(page_products)

        log_event(
            "api_fetch_products_page",
            "success",
            f"Fetched page with skip={skip}",
            extra={"page_count": len(page_products), "running_total": len(all_products)},
        )

        skip += page_size
        time.sleep(1)

    log_event(
        "api_fetch_products",
        "success",
        "Fetched all product pages successfully",
        extra={"total_products": len(all_products)},
    )

    return {"products": all_products, "total": len(all_products)}

def fetch_categories():
    log_event("api_fetch_categories", "started", "Fetching categories from DummyJSON")
    categories = fetch_json(CATEGORIES_ENDPOINT, timeout=REQUEST_TIMEOUT, max_attempts=MAX_ATTEMPTS)
    log_event(
        "api_fetch_categories",
        "success",
        "Fetched categories successfully",
        extra={"category_count": len(categories) if isinstance(categories, list) else 0},
    )
    return categories

# -----------------------------
# 5) HELPERS
# -----------------------------
def save_raw_json(obj, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def add_ingestion_metadata(df, source_file, source_mode):
    out = df.copy()
    out["source_system"] = SOURCE_NAME
    out["source_file"] = source_file
    out["source_mode"] = source_mode
    out["batch_id"] = BATCH_ID
    out["ingestion_ts"] = INGESTION_TS
    return out

def categories_to_dataframe(categories):
    if not isinstance(categories, list):
        return pd.DataFrame(columns=["category"])

    if len(categories) == 0:
        return pd.DataFrame(columns=["category"])

    if isinstance(categories[0], dict):
        return pd.json_normalize(categories)
    return pd.DataFrame({"category": categories})

# -----------------------------
# 6) MAIN
# -----------------------------
def main():
    manifest = {
        "source": SOURCE_NAME,
        "batch_id": BATCH_ID,
        "ingestion_ts": INGESTION_TS,
        "raw_output_dir": str(RAW_DIR),
        "bronze_output_dir": str(BRONZE_DIR),
        "files": [],
    }

    source_mode = "api"

    try:
        products_payload = fetch_all_products(page_size=PAGE_SIZE)
        categories_payload = fetch_categories()
    except Exception as e:
        log_event("api_ingestion", "failed", "Live API ingestion failed", extra={"error": str(e)})

        if not USE_OFFLINE_FALLBACK:
            raise

        source_mode = "offline_fallback"
        products_payload = get_fallback_products()
        categories_payload = get_fallback_categories()

        log_event(
            "api_ingestion",
            "fallback",
            "Switched to offline fallback payload",
            extra={"fallback_products": len(products_payload.get('products', []))},
        )

    products_raw_path = RAW_DIR / "products_raw.json"
    categories_raw_path = RAW_DIR / "categories_raw.json"

    save_raw_json(products_payload, products_raw_path)
    save_raw_json(categories_payload, categories_raw_path)

    manifest["files"].append(
        {
            "file_name": "products_raw.json",
            "raw_path": str(products_raw_path),
            "record_count": len(products_payload.get("products", [])),
            "source_mode": source_mode,
            "saved_at": datetime.now(timezone.utc).isoformat(),
        }
    )

    manifest["files"].append(
        {
            "file_name": "categories_raw.json",
            "raw_path": str(categories_raw_path),
            "record_count": len(categories_payload) if isinstance(categories_payload, list) else 0,
            "source_mode": source_mode,
            "saved_at": datetime.now(timezone.utc).isoformat(),
        }
    )

    products = products_payload.get("products", [])
    products_df = pd.json_normalize(products, sep="_")
    products_df = add_ingestion_metadata(products_df, "products_raw.json", source_mode)

    categories_df = categories_to_dataframe(categories_payload)
    categories_df = add_ingestion_metadata(categories_df, "categories_raw.json", source_mode)

    products_parquet = BRONZE_DIR / "products.parquet"
    categories_parquet = BRONZE_DIR / "categories.parquet"
    summary_csv = BRONZE_DIR / "ingestion_summary.csv"

    products_df.to_parquet(products_parquet, index=False)
    categories_df.to_parquet(categories_parquet, index=False)

    summary_df = pd.DataFrame(
        [
            {
                "dataset": "products",
                "rows": len(products_df),
                "columns": len(products_df.columns),
                "output_path": str(products_parquet),
                "source_mode": source_mode,
            },
            {
                "dataset": "categories",
                "rows": len(categories_df),
                "columns": len(categories_df.columns),
                "output_path": str(categories_parquet),
                "source_mode": source_mode,
            },
        ]
    )
    summary_df.to_csv(summary_csv, index=False)

    manifest["bronze_outputs"] = [
        str(products_parquet),
        str(categories_parquet),
        str(summary_csv),
    ]
    manifest["source_mode"] = source_mode

    with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    log_event(
        "pipeline",
        "success",
        "DummyJSON ingestion completed successfully",
        extra={
            "source_mode": source_mode,
            "manifest_file": str(MANIFEST_FILE),
            "products_rows": len(products_df),
            "categories_rows": len(categories_df),
        },
    )

    print("DummyJSON ingestion completed successfully")
    print(f"Source mode: {source_mode}")
    print(f"Raw dir: {RAW_DIR}")
    print(f"Bronze dir: {BRONZE_DIR}")
    print(f"Manifest: {MANIFEST_FILE}")
    print("\nSummary:")
    print(summary_df)

if __name__ == "__main__":
    main()


DummyJSON ingestion completed successfully
Source mode: api
Raw dir: C:\Users\barath\recomart-pipeline\data\raw\dummyjson\load_date=2026-04-30\load_hour=17
Bronze dir: C:\Users\barath\recomart-pipeline\data\bronze\dummyjson
Manifest: C:\Users\barath\recomart-pipeline\metadata\dummyjson_manifest_20260430T171416Z.json

Summary:
      dataset  rows  columns  \
0    products   194       32   
1  categories    24        8   

                                         output_path source_mode  
0  C:\Users\barath\recomart-pipeline\data\bronze\...         api  
1  C:\Users\barath\recomart-pipeline\data\bronze\...         api  
